In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#import prince
import statsmodels.api as sm
import statsmodels.formula.api as smf
import itertools
import scipy.stats as stats
from sklearn.model_selection import train_test_split

In [39]:
#loading the training data
file_path_train = "fraudTrain.csv"

fraud_train = pd.read_csv(file_path_train)

fraud_train.rename(columns={fraud_train.columns[0]: "id"}, inplace=True)

fraud_train = fraud_train.set_index(fraud_train.columns[0])


In [40]:
#Loading the tesing data
file_path_valid = "fraudTest.csv"

fraud_valid = pd.read_csv(file_path_valid)

fraud_valid.rename(columns={fraud_valid.columns[0]: "id"}, inplace=True)

fraud_valid = fraud_valid.set_index(fraud_valid.columns[0])

In [41]:
#Training and validation split
training_data, testing_data = train_test_split(fraud_train, test_size=0.2, random_state=1376)

print("Training set shape:")
print(training_data.shape)
print("Testing set shape:")
print(testing_data.shape) 

Training set shape:
(1037340, 22)
Testing set shape:
(259335, 22)


In [42]:
#Creating a population col on the state level by combining the pop's of the cities
#Honestly not sure if this is a good way of doing it, might be better off grabbing a diff data set and merging it in
city_pop = (
    training_data[["city", "state", "city_pop"]]
    .drop_duplicates()
)

state_pop = (
    city_pop.groupby("state")["city_pop"]
            .sum()
)

training_data["state_pop"] = training_data["state"].map(state_pop)

In [43]:
training_data.dtypes

trans_date_trans_time     object
cc_num                     int64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
state_pop                  int64
dtype: object

In [44]:
#freq table for cat with % 
for col in training_data.select_dtypes(include=["object", "category"]).columns:
    
    freq = training_data[col].value_counts(dropna=False)
    percent = training_data[col].value_counts(normalize=True, dropna=False) * 100
    
    table = pd.DataFrame({
        "Frequency": freq,
        "Percent": percent
    })
    
    print(f"\n{col}")
    print(table)


trans_date_trans_time
                       Frequency   Percent
trans_date_trans_time                     
2020-06-01 01:37:47            4  0.000386
2019-12-11 16:01:10            3  0.000289
2019-12-15 15:39:34            3  0.000289
2019-12-29 16:28:32            3  0.000289
2020-03-30 23:35:21            3  0.000289
...                          ...       ...
2019-07-16 20:40:55            1  0.000096
2019-04-22 09:21:28            1  0.000096
2019-02-26 17:20:30            1  0.000096
2019-12-15 20:28:20            1  0.000096
2019-11-26 13:02:23            1  0.000096

[1023316 rows x 2 columns]

merchant
                                       Frequency   Percent
merchant                                                  
fraud_Kilback LLC                           3503  0.337691
fraud_Cormier LLC                           2926  0.282068
fraud_Schumm PLC                            2915  0.281007
fraud_Boyer PLC                             2793  0.269246
fraud_Kuhn LLC            

In [45]:
#just a look at cols
print("Number of unique cities:")
print(training_data["city"].nunique())
print("Number of unique zip codes:")
print(training_data["zip"].nunique())
print("Number of unique merchants:")
print(training_data["merchant"].nunique())
print("Number of unique purchase catagories:")
print(training_data["category"].nunique())
print("Unique purchase catagories:")
print(training_data["category"].unique())
print("Number of included states:")
print(training_data["state"].nunique())
print("list of states:")
print(training_data["state"].unique())
print("51 states because Washington DC is included")
print("Min transaction ammount:")
print(training_data["amt"].min())
print("Mean transaction ammount:")
print(training_data["amt"].mean())
print("Median transaction ammount:")
print(training_data["amt"].median())
print("Max transaction ammount:")
print(training_data["amt"].max())
print("Number of frauds:")
print(training_data["is_fraud"].value_counts())
print("Ratio of frauds in %:")
print((6039/(6039 + 1031301))*100)
print("It looks like less than 1% of the total transactions are fraudulent")
print("Number of frauds in testing set:")
print(testing_data["is_fraud"].value_counts())
print("Testing ratio of frauds in %:")
print((1467/(1467 + 257868))*100)
print("Just wanting to make sure the testing set reflects the training set, and it does")



Number of unique cities:
894
Number of unique zip codes:
970
Number of unique merchants:
693
Number of unique purchase catagories:
14
Unique purchase catagories:
['kids_pets' 'gas_transport' 'grocery_net' 'home' 'food_dining' 'misc_pos'
 'misc_net' 'travel' 'shopping_net' 'grocery_pos' 'shopping_pos'
 'entertainment' 'personal_care' 'health_fitness']
Number of included states:
51
list of states:
['WA' 'LA' 'NY' 'SC' 'MA' 'PA' 'NJ' 'WI' 'MN' 'MI' 'AR' 'OR' 'TX' 'AL'
 'MO' 'IN' 'NV' 'OH' 'FL' 'ME' 'MD' 'NE' 'NM' 'KY' 'KS' 'VT' 'NH' 'IL'
 'NC' 'AZ' 'IA' 'SD' 'DC' 'GA' 'MS' 'OK' 'VA' 'ND' 'WY' 'CA' 'UT' 'MT'
 'WV' 'CO' 'CT' 'TN' 'ID' 'HI' 'AK' 'RI' 'DE']
51 states because Washington DC is included
Min transaction ammount:
1.0
Mean transaction ammount:
70.34589298590626
Median transaction ammount:
47.53
Max transaction ammount:
28948.9
Number of frauds:
is_fraud
0    1031301
1       6039
Name: count, dtype: int64
Ratio of frauds in %:
0.5821620683671699
It looks like less than 1% of the tot